In [1]:
# 🧠 C. elegans Connectome Overnight Pipeline (VGAE-Free)
# Requirements: Python 3.8+, Pandas, NetworkX, Matplotlib, Plotly, Seaborn, PyTorch, torch_geometric

# 📦 Imports
import os
import random
import torch
import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.stats import entropy, hypergeom
import seaborn as sns
import logging
from datetime import datetime
from torch_geometric.utils import from_networkx
import pickle  # Optional, not used now

# 🛠️ Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()

# ⚙️ Config
CONFIG = {
    'symbolic_neuron_tags': ['Apollo', 'Zeus', 'Hephaestus'],
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'output_dir': './vgae_outputs/overnight_analysis/',
    'log_dir': './logs/',
    'data_dir': './data/connectivity/processed/',
    'master_csv': 'neuron_master.csv',
    'edges_csv': 'merged_edges_with_relatedness.csv',
    'k_values': [1, 2, 3],
    'min_nodes': 5,
}

# ✅ Output directories
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['log_dir'], exist_ok=True)

# 📝 Logging
logging.basicConfig(
    filename=f"{CONFIG['log_dir']}/connectome_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
print(f"🧠 Using device: {CONFIG['device']}")

# 📂 Load data
df_edges = pd.read_csv(os.path.join(CONFIG['data_dir'], CONFIG['edges_csv']))
df_master = pd.read_csv(CONFIG['master_csv'])

# 🔗 Build full directed graph
G_full = nx.from_pandas_edgelist(df_edges, source="Source", target="Target", edge_attr=True, create_using=nx.DiGraph())
full_data = from_networkx(G_full)
neuron_list = sorted(set(G_full.nodes))
neuron_idx = {n: i for i, n in enumerate(neuron_list)}
full_data.x = torch.eye(len(neuron_list))

# 🧬 Prepare motif matrix
motif_cols = [col for col in df_master.columns if col.startswith('motif_')]
if not motif_cols:
    motif_cols = ['motif_oscillator', 'motif_loop', 'motif_feedforward']
    df_master[motif_cols] = np.random.rand(len(df_master), len(motif_cols))  # Placeholder

motif_matrix = df_master.set_index('Neuron')[motif_cols]

# ✅ Task 1: Motif-Driven Functional Divergence
def motif_divergence_analysis():
    print("▶️ Task 1: Motif Divergence Analysis")
    lineage_depths = pd.Series(np.random.randint(1, 5, size=len(df_master)), index=df_master['Neuron'])
    grouped = pd.concat([df_master.set_index('Neuron')['Type'], lineage_depths.rename('depth')], axis=1)
    unique_depths = sorted(lineage_depths.unique())
    divergence_results = []

    for neuron_type in df_master['Type'].unique():
        for depth in unique_depths:
            mask = (grouped['Type'] == neuron_type) & (grouped['depth'] == depth)
            if mask.sum() < 2:
                continue
            motif_subset = motif_matrix[mask]
            motif_mean = motif_subset.mean() + 1e-10
            kl_div = entropy(motif_subset.values.T, motif_mean.values[:, None]).mean()
            divergence_results.append({'type': neuron_type, 'depth': depth, 'kl_divergence': kl_div})

    divergence_df = pd.DataFrame(divergence_results)
    divergence_df.to_csv(os.path.join(CONFIG['output_dir'], 'motif_divergence.csv'))

    plt.figure(figsize=(10, 6))
    pivot_table = divergence_df.pivot(index='type', columns='depth', values='kl_divergence')
    sns.heatmap(pivot_table, annot=True, cmap='viridis')
    plt.title("Motif Divergence by Neuron Type and Lineage Depth")
    plt.savefig(os.path.join(CONFIG['output_dir'], 'motif_divergence_heatmap.png'))
    plt.close()

    # Symbolic node overlay
    symbolic_neurons = df_master[df_master['SymbolicRole'].isin(CONFIG['symbolic_neuron_tags'])]['Neuron']
    G_symbolic = G_full.subgraph(symbolic_neurons)
    node_colors = []
    for n in G_symbolic.nodes:
        neuron_type = df_master[df_master['Neuron'] == n]['Type'].iloc[0]
        kl_val = divergence_df[divergence_df['type'] == neuron_type]['kl_divergence'].mean()
        node_colors.append(kl_val)
    pos = nx.spring_layout(G_symbolic)
    plt.figure(figsize=(12, 8))
    nx.draw(G_symbolic, pos, node_color=node_colors, cmap='viridis', node_size=80)
    plt.title("Symbolic Neurons with Motif Divergence")
    plt.savefig(os.path.join(CONFIG['output_dir'], 'symbolic_divergence_graph.png'))
    plt.close()

# ✅ Task 2: Synthetic Lesion Impact
def lesion_impact_ranking():
    print("▶️ Task 2: Lesion Impact Ranking")
    base_clustering = nx.average_clustering(G_full)
    base_centrality = nx.betweenness_centrality(G_full)
    base_motif_counts = motif_matrix.sum()

    impact_results = []
    for neuron in tqdm(G_full.nodes, desc="Simulating lesions"):
        G_temp = G_full.copy()
        G_temp.remove_node(neuron)

        new_clustering = nx.average_clustering(G_temp)
        new_centrality = nx.betweenness_centrality(G_temp)
        new_motif_counts = motif_matrix.drop(neuron, errors='ignore').sum()

        impact_results.append({
            'neuron': neuron,
            'clustering_impact': abs(base_clustering - new_clustering),
            'centrality_impact': sum(abs(base_centrality[n] - new_centrality.get(n, 0)) for n in base_centrality),
            'motif_impact': abs(base_motif_counts - new_motif_counts).mean(),
            'type': df_master[df_master['Neuron'] == neuron]['Type'].iloc[0] if neuron in df_master['Neuron'].values else 'unknown',
            'symbolic_role': df_master[df_master['Neuron'] == neuron]['SymbolicRole'].iloc[0] if neuron in df_master['Neuron'].values else 'unknown'
        })

    impact_df = pd.DataFrame(impact_results)
    impact_df['total_impact'] = impact_df[['clustering_impact', 'centrality_impact', 'motif_impact']].sum(axis=1)
    impact_df.sort_values('total_impact', ascending=False).to_csv(os.path.join(CONFIG['output_dir'], 'lesion_impact_ranking.csv'))

    top_neurons = impact_df[impact_df['symbolic_role'].isin(CONFIG['symbolic_neuron_tags'])].nlargest(20, 'total_impact')
    G_sub = G_full.subgraph(top_neurons['neuron'])
    pos = nx.spring_layout(G_full)
    plt.figure(figsize=(12, 8))
    nx.draw(G_sub, pos, node_color=[impact_df[impact_df['neuron'] == n]['total_impact'].iloc[0] for n in G_sub.nodes],
            cmap='plasma', node_size=100)
    plt.title("Top 20 High-Impact Symbolic Neurons")
    plt.savefig(os.path.join(CONFIG['output_dir'], 'lesion_impact_symbolic_network.png'))
    plt.close()

# ✅ Task 3: Motif Enrichment Flow
def motif_enrichment_flow():
    print("▶️ Task 3: Motif Enrichment Flow")
    lineage_depths = pd.Series(np.random.randint(1, 5, size=len(df_master)), index=df_master['Neuron'])
    unique_depths = sorted(lineage_depths.unique())
    enrichment_results = []

    total_neurons = len(motif_matrix)
    for motif in motif_matrix.columns:
        for depth in unique_depths:
            depth_neurons = lineage_depths[lineage_depths == depth].index
            motif_count = motif_matrix.loc[depth_neurons, motif].sum()
            total_motif = motif_matrix[motif].sum()
            depth_size = len(depth_neurons)
            pval = hypergeom.sf(motif_count - 1, total_neurons, total_motif, depth_size)
            enrichment_results.append({
                'motif': motif, 'depth': depth, 'enrichment_pval': pval, 'motif_count': motif_count
            })

    enrichment_df = pd.DataFrame(enrichment_results)
    enrichment_df.to_csv(os.path.join(CONFIG['output_dir'], 'motif_enrichment.csv'))

    # Sankey
    nodes = list(map(str, unique_depths)) + motif_matrix.columns.tolist() + CONFIG['symbolic_neuron_tags']
    node_indices = {n: i for i, n in enumerate(nodes)}
    source, target, value = [], [], []

    for _, row in enrichment_df.iterrows():
        if row['enrichment_pval'] < 0.05:
            source.append(node_indices[str(row['depth'])])
            target.append(node_indices[row['motif']])
            value.append(row['motif_count'])

    for motif in motif_matrix.columns:
        for role in CONFIG['symbolic_neuron_tags']:
            role_neurons = df_master[df_master['SymbolicRole'] == role]['Neuron']
            motif_count = motif_matrix.loc[role_neurons, motif].sum()
            if motif_count > 0:
                source.append(node_indices[motif])
                target.append(node_indices[role])
                value.append(motif_count)

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, line=dict(color="black", width=0.5), label=nodes),
        link=dict(source=source, target=target, value=value)
    )])
    fig.update_layout(title_text="Motif Enrichment Flow", font_size=10)
    fig.write_html(os.path.join(CONFIG['output_dir'], 'motif_enrichment_sankey.html'))

# ▶️ Run all
motif_divergence_analysis()
lesion_impact_ranking()
motif_enrichment_flow()
print("✅ All tasks completed successfully.")


/home/rohit/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


🧠 Using device: cuda
▶️ Task 1: Motif Divergence Analysis
▶️ Task 2: Lesion Impact Ranking


Simulating lesions: 100%|██████████| 1533/1533 [26:48<00:00,  1.05s/it]


▶️ Task 3: Motif Enrichment Flow
✅ All tasks completed successfully.
